# v2 Persona Vector Pipeline — Method Comparison

Pure numpy/sklearn on the cache written by `01_extract_activations.ipynb` — no GPU needed, runs anywhere (including a laptop) as long as `PV2_CACHE_DIR` points at that cache (a local copy, or a mounted Drive folder).

Produces, per model: accuracy + bootstrap CI for all three methods on sub-test A and sub-test B's reportable neutral arm, the A/B gap (the spec's win condition), the TF-IDF floor, a random-direction null check, the length-correlation flag for every reported number, and the probe-vs-cosine comparison for whichever method has the smallest A/B gap. Models whose cache isn't present yet are skipped with a clear message, not silently omitted — this notebook is meant to be re-run incrementally as `01_extract_activations.ipynb` finishes each model.

In [ ]:
import os, sys, pathlib

PIPELINE_DIR = pathlib.Path.cwd()
if not (PIPELINE_DIR / "src").exists():
    # allow running from repo root or notebooks/ as well as Pipeline_v2/
    for candidate in [pathlib.Path.cwd().parent, pathlib.Path.cwd() / "Pipeline_v2"]:
        if (candidate / "src").exists():
            PIPELINE_DIR = candidate
            break
sys.path.insert(0, str(PIPELINE_DIR))

# If the cache lives somewhere other than Pipeline_v2/cache (e.g. a Drive-mounted
# folder from the extraction notebook), set PV2_CACHE_DIR BEFORE this import.
# os.environ["PV2_CACHE_DIR"] = "/content/drive/MyDrive/persona_vector_v2_cache"

from src.config import CACHE_DIR, MODEL_REGISTRY
from src import activation_store

print("cache dir:", CACHE_DIR)
available_models = [k for k in MODEL_REGISTRY if (CACHE_DIR / k).exists()]
print("models with a cache present:", available_models or "(none yet)")

## TF-IDF baseline (spec requirement 4) — model-independent, computed once

In [ ]:
import numpy as np
from src import splits
from src.methods import tfidf_baseline

splits_df = splits.load_frozen_splits()
text_by_id = {row.prompt_id: row.text for row in splits.load_combined_rows()}

def _ids_labels(split_name):
    subset = splits_df[splits_df["split"] == split_name]
    return subset["PromptID"].tolist(), subset["label"].tolist()

train_ids, train_labels = _ids_labels("train")
test_ids, test_labels = _ids_labels("test")
all_ids, all_labels = splits_df["PromptID"].tolist(), splits_df["label"].tolist()

tfidf_single = tfidf_baseline.fit_and_score_single(
    [text_by_id[i] for i in train_ids], np.array(train_labels),
    [text_by_id[i] for i in test_ids], np.array(test_labels),
)
tfidf_cv = tfidf_baseline.cross_val_score_5fold([text_by_id[i] for i in all_ids], np.array(all_labels))

print(f"TF-IDF single-fit accuracy: {tfidf_single['accuracy']:.4f} (n_train={tfidf_single['n_train']}, n_test={tfidf_single['n_test']})")
print(f"TF-IDF 5-fold CV accuracy:  {tfidf_cv['mean_accuracy']:.4f} +/- {tfidf_cv['std_accuracy']:.4f}")
print("(v1's reference numbers were 66.39% single-fit / 76.17% +/- 2.07 CV -- not expected to match exactly, different dataset, but should be in the same neighborhood as a sanity check)")

## Length-balance note (spec deliverable 5)

In [ ]:
lengths = np.array([len(text_by_id[i]) for i in all_ids])
is_harmful = np.array([1.0 if lbl == "harmful" else 0.0 for lbl in all_labels])
r_length_label = float(np.corrcoef(lengths, is_harmful)[0, 1])
print(f"r(length, label) across the combined 550-prompt set: {r_length_label:.3f}")
print("(manifests report 0.436 at authoring time -- this recomputes it directly from the frozen split's own text)")
print("Per spec, this makes centering/standardization/last-token pooling a hard requirement, not an optional ablation.")
print("Per-method, per-pooling-variant r(score, length) is reported below in each method's length_correlation block --")
print("any result with flagged=True should be treated as unreliable for reporting, the same way v1's mean-pooled results were.")

## Per-model method comparison: sub-test A, sub-test B, A/B gap, random-direction control

In [ ]:
import pandas as pd
from src.eval import subtest_a as subtest_a_mod
from src.eval import subtest_b as subtest_b_mod

all_results = {}       # model_key -> {method_name: FittedMethodResult}
all_b_results = {}     # model_key -> subtest_b results
all_gap_summaries = {} # model_key -> gap summary
rows = []

for model_key in available_models:
    formatting_variants = [d.name for d in (CACHE_DIR / model_key).iterdir() if d.is_dir() and d.name != "generation"]
    for formatting_variant in formatting_variants:
        print(f"=== {model_key} / {formatting_variant} ===")
        a_results = subtest_a_mod.run_subtest_a(CACHE_DIR, model_key, formatting_variant)
        b_results = subtest_b_mod.run_subtest_b(CACHE_DIR, model_key, formatting_variant, a_results)
        gap_summary = subtest_b_mod.summarize_a_b_gap(a_results, b_results)

        all_results[(model_key, formatting_variant)] = a_results
        all_b_results[(model_key, formatting_variant)] = b_results
        all_gap_summaries[(model_key, formatting_variant)] = gap_summary

        for method_name, fitted in a_results.items():
            gap = gap_summary.get(method_name, {})
            rows.append({
                "model": model_key, "formatting": formatting_variant, "method": method_name,
                "pooling_variant": fitted.pooling_variant, "layer": fitted.layer,
                "subtest_a_accuracy": fitted.test_result["accuracy"],
                "subtest_a_ci_low": fitted.test_result["ci_low"],
                "subtest_a_ci_high": fitted.test_result["ci_high"],
                "subtest_a_length_flagged": fitted.test_result["length_correlation"]["flagged"],
                "subtest_b_neutral_accuracy": gap.get("subtest_b_neutral_arm_accuracy"),
                "subtest_b_neutral_ci": gap.get("subtest_b_neutral_arm_ci"),
                "a_b_gap": gap.get("a_b_gap"),
                "note": fitted.note,
            })

results_df = pd.DataFrame(rows)
results_df

## Win condition: smallest A/B gap with competitive absolute accuracy on A

Per spec, this is decided per (model, formatting_variant) — do not presuppose Method 3 wins.

In [ ]:
COMPETITIVE_A_THRESHOLD = 0.60  # edit if a different bar for "competitive" is wanted

winners = []
for (model_key, formatting_variant), gap_summary in all_gap_summaries.items():
    competitive = {m: g for m, g in gap_summary.items() if g["subtest_a_accuracy"] >= COMPETITIVE_A_THRESHOLD}
    pool = competitive or gap_summary
    if not pool:
        continue
    winner_name = min(pool, key=lambda m: abs(pool[m]["a_b_gap"]))
    winners.append({
        "model": model_key, "formatting": formatting_variant, "winning_method": winner_name,
        **pool[winner_name],
    })

winners_df = pd.DataFrame(winners)
winners_df

## Probe vs. cosine/mean-diff (the second open method question)

Run for whichever method won each (model, formatting_variant) above, at the layer/pooling variant that method already selected on validation. Reported as a comparison, not a replacement.

In [ ]:
from src.methods import probe as probe_mod

probe_rows = []
for winner in winners:
    key = (winner["model"], winner["formatting"])
    fitted = all_results[key][winner["winning_method"]]

    splits_df_local = splits.load_frozen_splits()
    train_ids_l, train_labels_l = _ids_labels("train")
    test_ids_l, test_labels_l = _ids_labels("test")

    train_matrix = activation_store.load_layer_matrix(
        CACHE_DIR, winner["model"], winner["formatting"], fitted.pooling_variant, fitted.layer, train_ids_l
    )
    test_matrix = activation_store.load_layer_matrix(
        CACHE_DIR, winner["model"], winner["formatting"], fitted.pooling_variant, fitted.layer, test_ids_l
    )
    probe_result = probe_mod.fit_and_score_probe(
        train_matrix, np.array(train_labels_l), test_matrix, np.array(test_labels_l)
    )
    probe_rows.append({
        "model": winner["model"], "formatting": winner["formatting"], "method": winner["winning_method"],
        "cosine_meandiff_accuracy": winner["subtest_a_accuracy"],
        "probe_accuracy": probe_result["accuracy"],
    })

pd.DataFrame(probe_rows)

## Random-direction null check (spec requirement 4)

In [ ]:
control_rows = []
for (model_key, formatting_variant), a_results in all_results.items():
    for method_name, fitted in a_results.items():
        control = subtest_a_mod.evaluate_random_direction_control(CACHE_DIR, model_key, formatting_variant, fitted)
        control_rows.append({
            "model": model_key, "formatting": formatting_variant, "method": method_name,
            "fitted_accuracy": control["fitted_method_test_accuracy"],
            "random_direction_mean_accuracy": control["mean_accuracy"],
            "random_direction_range": (control["min_accuracy"], control["max_accuracy"]),
        })

pd.DataFrame(control_rows)

## Sub-test B harmful arm — qualitative only, N=3, never a headline number

Per data/subtest_b_MANIFEST.md and explicit instruction: printed separately, never merged into the table above, never bootstrap-CI'd.

In [ ]:
for (model_key, formatting_variant), b_results in all_b_results.items():
    for method_name, arms in b_results.items():
        qual = arms["harmful_arm_qualitative"]
        n_correct = sum(item["correct"] for item in qual["per_item"])
        print(f"{model_key} / {formatting_variant} / {method_name}: {n_correct}/{len(qual['per_item'])} correct on "
              f"the N={qual['n_pairs']} verified-clean harmful arm -- {qual['note']}")

In [ ]:
# Save the deliverable table.
output_path = pathlib.Path("method_comparison_results.csv")
results_df.to_csv(output_path, index=False)
print("wrote", output_path.resolve())